# Using Python to Verify the AI Output

In [4]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [5]:
# Create timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print("Selena Chik")

Current Time:09_27_2026_0308PM
Selena Chik


In [7]:
data2025 = pd.read_csv("2025.csv")
data2026 = pd.read_csv("2026PartYear.csv")

print(data2025.shape)
print(data2026.shape)

print(data2025.columns)

(6656, 16)
(4096, 16)
Index(['Report', 'Timeframe', 'Period', 'Type', 'Geography',
       'Current Year Week Ending', 'ASP Current Year', 'Total Bulk and Bags',
       '4046', '4225', '4770', 'TotalBagged', 'Sml Bagged', 'Lrg Bagged',
       'X-Lrg Bagged', 'Unknown Bag Size'],
      dtype='str')


In [8]:
# Validate 2026 Data Period
 
print("2026 Periods:", data2026["Period"].min(), "to", data2026["Period"].max())
print("2026 Number of Periods:", data2026["Period"].nunique())
 
print("2026 First Date:", data2026["Current Year Week Ending"].min())
print("2026 Last Date:", data2026["Current Year Week Ending"].max())

2026 Periods: 1 to 32
2026 Number of Periods: 32
2026 First Date: 2026/01/04
2026 Last Date: 2026/08/09


In [9]:
# Validate Filtering 2025 to the same 32 periods as 2026
data2025_ytd = data2025[data2025["Period"] <= 32].copy()
 
print("2025 Periods:", data2025_ytd["Period"].min(), "to", data2025_ytd["Period"].max())
print("2025 Number of Periods:", data2025_ytd["Period"].nunique())
 
print("2025 First Date:", data2025_ytd["Current Year Week Ending"].min())
print("2025 Last Date:", data2025_ytd["Current Year Week Ending"].max())

2025 Periods: 1 to 32
2025 Number of Periods: 32
2025 First Date: 2025/01/05
2025 Last Date: 2025/08/10


In [24]:
# Validate Total U.S. Volume
 
# Filter for Total U.S.
us2025 = data2025_ytd[data2025_ytd["Geography"] == "Total U.S."]
us2026 = data2026[data2026["Geography"] == "Total U.S."]

# Calculate total volume
volume2025 = us2025["Total Bulk and Bags"].sum()
volume2026 = us2026["Total Bulk and Bags"].sum()

print("2025 Total U.S. Volume:", volume2025)
print("2026 Total U.S. Volume:", volume2026)
print("Volume Change:", round((volume2026 - volume2025) / volume2025 * 100, 1), "%")

2025 Total U.S. Volume: 1870698865.488
2026 Total U.S. Volume: 2125800177.369
Volume Change: 13.6 %


In [23]:
# Validate Total U.S. Weighted ASP
 
# Calculate volume-weighted ASP for Total U.S.
weighted_asp2025 = (
    (us2025["ASP Current Year"] * us2025["Total Bulk and Bags"]).sum()
    / volume2025)
 
weighted_asp2026 = (
    (us2026["ASP Current Year"] * us2026["Total Bulk and Bags"]).sum()
    / volume2026)

print("2025 Weighted ASP:", round(weighted_asp2025, 2))
print("2026 Weighted ASP:", round(weighted_asp2026, 2))
print("ASP Change:", round((weighted_asp2026 - weighted_asp2025) / weighted_asp2025 * 100, 1), "%")

2025 Weighted ASP: 1.25
2026 Weighted ASP: 1.01
ASP Change: -19.1 %


In [20]:
# Validate Volume by Avocado Type
 
# Calculate total volume by avocado type
type_volume2025 = us2025.groupby("Type")["Total Bulk and Bags"].sum()
type_volume2026 = us2026.groupby("Type")["Total Bulk and Bags"].sum()

print("2025 Volume by Type:")
print(type_volume2025)
 
print("\n2026 Volume by Type:")
print(type_volume2026)
 
print("\nVolume Change by Type:")
print(round((type_volume2026 - type_volume2025) / type_volume2025 * 100, 1))

2025 Volume by Type:
Type
Conventional    1.764839e+09
Organic         1.058595e+08
Name: Total Bulk and Bags, dtype: float64

2026 Volume by Type:
Type
Conventional    2.020685e+09
Organic         1.051156e+08
Name: Total Bulk and Bags, dtype: float64

Volume Change by Type:
Type
Conventional    14.5
Organic         -0.7
Name: Total Bulk and Bags, dtype: float64


In [18]:
# Validate Volume Share by Avocado Type
 
# Calculate percentage of total volume by avocado type
type_share2025 = type_volume2025 / volume2025 * 100
type_share2026 = type_volume2026 / volume2026 * 100
 
print("2025 Volume Share by Type:")
print(type_share2025.round(1))
 
print("\n2026 Volume Share by Type:")
print(type_share2026.round(1))

2025 Volume Share by Type:
Type
Conventional    94.3
Organic          5.7
Name: Total Bulk and Bags, dtype: float64

2026 Volume Share by Type:
Type
Conventional    95.1
Organic          4.9
Name: Total Bulk and Bags, dtype: float64


In [22]:
# Validate Weighted ASP by Avocado Type
 
# Calculate volume-weighted ASP by avocado type
weighted_type_asp2025 = (
    (us2025["ASP Current Year"] * us2025["Total Bulk and Bags"])
    .groupby(us2025["Type"]).sum()
    / type_volume2025)
 
weighted_type_asp2026 = (
    (us2026["ASP Current Year"] * us2026["Total Bulk and Bags"])
    .groupby(us2026["Type"]).sum()
    / type_volume2026)

print("2025 Weighted ASP by Type:")
print(weighted_type_asp2025.round(2))
 
print("\n2026 Weighted ASP by Type:")
print(weighted_type_asp2026.round(2))
 
print("\nWeighted ASP Change by Type:")
print(round((weighted_type_asp2026 - weighted_type_asp2025) / weighted_type_asp2025 * 100, 1))

2025 Weighted ASP by Type:
Type
Conventional    1.22
Organic         1.74
dtype: float64

2026 Weighted ASP by Type:
Type
Conventional    0.98
Organic         1.56
dtype: float64

Weighted ASP Change by Type:
Type
Conventional   -19.4
Organic        -10.3
dtype: float64


In [25]:
# Validate Organic Premium Over Conventional
 
premium2025 = (weighted_type_asp2025["Organic"] / weighted_type_asp2025["Conventional"] - 1) * 100
premium2026 = (weighted_type_asp2026["Organic"] / weighted_type_asp2026["Conventional"] - 1) * 100
 
print("2025 Organic Premium over Conventional:", round(premium2025, 1), "%")
print("2026 Organic Premium over Conventional:", round(premium2026, 1), "%")

2025 Organic Premium over Conventional: 43.2 %
2026 Organic Premium over Conventional: 59.5 %


In [26]:
# Validate Monthly Conventional ASP Trend
 
us2025["Month"] = pd.to_datetime(us2025["Current Year Week Ending"]).dt.month
us2026["Month"] = pd.to_datetime(us2026["Current Year Week Ending"]).dt.month
 
conv2025 = us2025[us2025["Type"] == "Conventional"]
conv2026 = us2026[us2026["Type"] == "Conventional"]
 
monthly_asp2025 = (
    (conv2025["ASP Current Year"] * conv2025["Total Bulk and Bags"])
    .groupby(conv2025["Month"]).sum()
    / conv2025.groupby("Month")["Total Bulk and Bags"].sum())
 
monthly_asp2026 = (
    (conv2026["ASP Current Year"] * conv2026["Total Bulk and Bags"])
    .groupby(conv2026["Month"]).sum()
    / conv2026.groupby("Month")["Total Bulk and Bags"].sum())
 
monthly_compare = pd.DataFrame({"2025": monthly_asp2025, "2026": monthly_asp2026})
monthly_compare["pct_change"] = (
    (monthly_compare["2026"] - monthly_compare["2025"]) / monthly_compare["2025"] * 100)
 
print("Monthly Conventional Weighted ASP, 2025 vs 2026:")
print(monthly_compare.round(3))

Monthly Conventional Weighted ASP, 2025 vs 2026:
        2025   2026  pct_change
Month                          
1      1.163  0.924     -20.554
2      1.221  0.903     -26.043
3      1.248  0.919     -26.305
4      1.274  0.943     -25.953
5      1.223  0.955     -21.962
6      1.225  1.126      -8.078
7      1.179  1.096      -7.092
8      1.141  1.070      -6.173


In [30]:
# Validate Volume Change by Geographic Market
 
# Define major geographic markets
major_markets = [
    "California",
    "Great Lakes",
    "Midsouth",
    "Northeast",
    "Plains",
    "South Central",
    "Southeast",
    "West"]

In [32]:
# Validate Los Angeles and New York Market Rankings
 
# Exclude Total U.S. and major geographic markets
exclude_markets = major_markets + ["Total U.S."]
 
individual2025 = data2025_ytd[
    ~data2025_ytd["Geography"].isin(exclude_markets)]
 
individual2026 = data2026[
    ~data2026["Geography"].isin(exclude_markets)]

In [34]:
# Calculate volume by individual market (Conventional, to match memo's regional table)
individual2025_conv = individual2025[individual2025["Type"] == "Conventional"]
individual2026_conv = individual2026[individual2026["Type"] == "Conventional"]
 
individual_volume2025 = (
    individual2025_conv.groupby("Geography")["Total Bulk and Bags"].sum())
individual_volume2026 = (
    individual2026_conv.groupby("Geography")["Total Bulk and Bags"].sum())
 
print("Largest 2025 Individual Markets (Conventional):")
print(individual_volume2025.sort_values(ascending=False).head(10))
 
print("\nLargest 2026 Individual Markets (Conventional):")
print(individual_volume2026.sort_values(ascending=False).head(10))

Largest 2025 Individual Markets (Conventional):
Geography
Los Angeles             1.285137e+08
New York                1.037835e+08
Dallas/Ft. Worth        5.651540e+07
Chicago                 5.583744e+07
Baltimore/Washington    5.532617e+07
Miami/FT. Lauderdale    5.094472e+07
Phoenix/Tucson          4.997036e+07
Houston                 3.824461e+07
San Francisco           3.823261e+07
Denver                  3.807994e+07
Name: Total Bulk and Bags, dtype: float64

Largest 2026 Individual Markets (Conventional):
Geography
Los Angeles             1.528505e+08
New York                1.196157e+08
Baltimore/Washington    6.664491e+07
Chicago                 6.432134e+07
Miami/FT. Lauderdale    6.302809e+07
Dallas/Ft. Worth        6.250426e+07
Phoenix/Tucson          5.547281e+07
Houston                 4.449212e+07
Denver                  4.206323e+07
San Francisco           4.168695e+07
Name: Total Bulk and Bags, dtype: float64


In [35]:
# Weighted ASP for the same top individual markets, 2026
individual_weighted_asp2026 = (
    (individual2026_conv["ASP Current Year"] * individual2026_conv["Total Bulk and Bags"])
    .groupby(individual2026_conv["Geography"]).sum()
    / individual_volume2026)
 
print("\n2026 Individual Market Weighted ASP (top 10 by volume):")
top10 = individual_volume2026.sort_values(ascending=False).head(10).index
print(individual_weighted_asp2026.loc[top10].round(3))


2026 Individual Market Weighted ASP (top 10 by volume):
Geography
Los Angeles             1.123
New York                1.019
Baltimore/Washington    0.944
Chicago                 0.891
Miami/FT. Lauderdale    0.867
Dallas/Ft. Worth        0.853
Phoenix/Tucson          0.993
Houston                 0.867
Denver                  1.139
San Francisco           1.281
dtype: float64
